# 02 — Pandas Native Bucketing Demo

Goal: pull raw telemetry rows from Postgres, then do bucketing/window logic in Pandas (not SQL).


## Imports


In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)


## Connection


In [2]:
DB_HOST='host.docker.internal'
DB_PORT=5432
DB_NAME='observability'
DB_USER='obs_user'
DB_PASS='obs_pass'
engine=create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}', pool_pre_ping=True)


## Load raw rows only (minimal SQL)


In [3]:
sql = '''
SELECT sampled_at, host, cpu_pct, mem_pct, region, env
FROM lab.telemetry_cpu_raw
WHERE sampled_at >= now() - interval '14 days'
'''
df = pd.read_sql_query(text(sql), engine)
df.head()


,sampled_at,host,cpu_pct,mem_pct,region,env
0,2026-04-30 09:30:43.007145+00:00,host09,60.58,81.17,us-west-2,prod
1,2026-04-30 09:31:01.836491+00:00,host20,31.44,56.46,us-west-2,prod
2,2026-04-30 09:31:04.051437+00:00,host54,54.40,88.07,us-east-1,prod
3,2026-04-30 09:31:10.060756+00:00,host14,54.88,68.57,eu-west-1,prod
4,2026-04-30 09:31:11.321173+00:00,host11,93.68,62.60,us-west-2,stage


## Pandas time bucketing


In [4]:
df['sampled_at'] = pd.to_datetime(df['sampled_at'], utc=True)
df['hour_bucket'] = df['sampled_at'].dt.floor('h')
df['day_bucket'] = df['sampled_at'].dt.floor('D')
df[['sampled_at','hour_bucket','day_bucket']].head()


,sampled_at,hour_bucket,day_bucket
0,2026-04-30 09:30:43.007145+00:00,2026-04-30 09:00:00+00:00,2026-04-30 00:00:00+00:00
1,2026-04-30 09:31:01.836491+00:00,2026-04-30 09:00:00+00:00,2026-04-30 00:00:00+00:00
2,2026-04-30 09:31:04.051437+00:00,2026-04-30 09:00:00+00:00,2026-04-30 00:00:00+00:00
3,2026-04-30 09:31:10.060756+00:00,2026-04-30 09:00:00+00:00,2026-04-30 00:00:00+00:00
4,2026-04-30 09:31:11.321173+00:00,2026-04-30 09:00:00+00:00,2026-04-30 00:00:00+00:00


## Pandas hourly aggregation


In [5]:
hourly = (
    df[df['env'] == 'prod']
      .groupby(['hour_bucket','host'], as_index=False)
      .agg(avg_cpu=('cpu_pct','mean'), peak_cpu=('cpu_pct','max'))
)
hourly['avg_cpu'] = hourly['avg_cpu'].round(2)
hourly.sort_values(['hour_bucket','host'], ascending=[False, True]).head(20)


,hour_bucket,host,avg_cpu,peak_cpu
23306,2026-05-14 00:00:00+00:00,host45,86.89,86.89
23239,2026-05-13 23:00:00+00:00,host02,75.85,91.69
23240,2026-05-13 23:00:00+00:00,host03,71.60,94.64
23241,2026-05-13 23:00:00+00:00,host04,65.24,79.62
23242,2026-05-13 23:00:00+00:00,host06,74.00,76.69
23243,2026-05-13 23:00:00+00:00,host07,63.75,75.88
23244,2026-05-13 23:00:00+00:00,host08,32.31,32.31
23245,2026-05-13 23:00:00+00:00,host09,48.57,63.31
23246,2026-05-13 23:00:00+00:00,host10,45.09,55.19
23247,2026-05-13 23:00:00+00:00,host12,41.85,53.83


## Pandas rolling 6-hour average by host


In [6]:
hourly2 = (
    df.groupby(['host','hour_bucket'], as_index=False)
      .agg(avg_cpu=('cpu_pct','mean'))
)
hourly2 = hourly2.sort_values(['host','hour_bucket'])
hourly2['rolling_6h_avg'] = (
    hourly2.groupby('host')['avg_cpu']
           .transform(lambda s: s.rolling(window=6, min_periods=1).mean())
)
hourly2['avg_cpu'] = hourly2['avg_cpu'].round(2)
hourly2['rolling_6h_avg'] = hourly2['rolling_6h_avg'].round(2)
hourly2.head(50)


,host,hour_bucket,avg_cpu,rolling_6h_avg
0,host01,2026-04-30 09:00:00+00:00,44.96,44.96
1,host01,2026-04-30 10:00:00+00:00,51.19,48.07
2,host01,2026-04-30 11:00:00+00:00,47.72,47.96
3,host01,2026-04-30 12:00:00+00:00,62.18,51.51
4,host01,2026-04-30 13:00:00+00:00,55.72,52.35
5,host01,2026-04-30 14:00:00+00:00,52.79,52.43
6,host01,2026-04-30 15:00:00+00:00,53.13,53.79
7,host01,2026-04-30 16:00:00+00:00,63.82,55.89
8,host01,2026-04-30 17:00:00+00:00,52.62,56.71
9,host01,2026-04-30 18:00:00+00:00,71.09,58.20
